Import ชุดคำสั่งที่จำเป็น

In [ ]:
import pandas as pd
import numpy as np

# for reading and displaying images
from skimage.io import imread
import matplotlib.pyplot as plt

# for creating validation set
from sklearn.model_selection import train_test_split

# for evaluating the model
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# PyTorch libraries and modules
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam, SGD


Data Loader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/Deep Learning/ThaiCharacter Dataset.zip" -d /content/dataset

# จากนั้นตั้งตัวแปร path สำหรับใช้ต่อในโค้ด:

DATA_PATH = "/content/dataset/round2"

In [ ]:
import os

DATA_PATH = "/content/dataset/round2"
classes = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))])

Data Loader + Data Augmentation (วนอ่านภาพจริงพร้อมเติมภาพให้คลาสที่ขาด)

In [ ]:
from skimage.transform import resize, rotate
from skimage.util import random_noise

def augment_image(img):
    angle = np.random.uniform(-15, 15)  # เทียบเท่า RandomRotation
    img_aug = rotate(img, angle, mode='edge')
    if np.random.rand() < 0.5:
        img_aug = random_noise(img_aug, var=0.005)  # เพิ่ม noise เบาๆ
    return img_aug

In [ ]:
MIN_SAMPLES_PER_CLASS = 50
IMG_SIZE = 64  # เพิ่มจาก 32 -> 64 กัน feature map ยุบเหลือ 1x1 ก่อนถึง fc ของ ResNet18

train_img = []
train_label = []

for cls in tqdm(classes):
    cls_folder = os.path.join(DATA_PATH, cls)
    img_files = [f for f in os.listdir(cls_folder) if f.lower().endswith('.jpg')]

    original_imgs = []
    for fname in img_files:
        img = imread(os.path.join(cls_folder, fname), as_gray=True)
        img = resize(img, (IMG_SIZE, IMG_SIZE), preserve_range=True)  # preserve_range=True ป้องกันการหารซ้ำ
        # skimage as_gray=True: ถ้าภาพต้นทางเป็น RGB จะแปลงผ่าน rgb2gray -> ได้ float ช่วง 0-1 อยู่แล้ว
        # ถ้าเป็น grayscale เดี่ยวอยู่แล้ว (เหมือนชุดนี้) จะคืน uint8 ช่วง 0-255 -> เช็ค max ก่อนหารกันหารซ้ำสอง
        if img.max() > 1.0:
            img /= 255.0
        original_imgs.append(img.astype('float32'))

    train_img.extend(original_imgs)
    train_label.extend([cls] * len(original_imgs))

    # เติมภาพด้วย Augmentation เฉพาะคลาสที่ขาด
    n_needed = MIN_SAMPLES_PER_CLASS - len(original_imgs)
    if n_needed > 0:
        for i in range(n_needed):
            base_img = original_imgs[i % len(original_imgs)]
            train_img.append(augment_image(base_img).astype('float32'))
            train_label.append(cls)

train_x = np.array(train_img)
train_y = np.array(train_label)
print(train_x.shape)

Training & Validating Set Generation

In [ ]:
# แปลง label เป็น class index ก่อน split (ต้องทำก่อน ไม่งั้น val_y จะไม่ได้แปลงด้วย)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}  # classes จากตอน Data Loader
train_y = np.array([class_to_idx[l] for l in train_y])

# stratify=train_y กัน val set สุ่มไม่สมดุลระหว่าง 72 คลาส (มีบางคลาสตัวอย่างน้อย)
train_x, val_x, train_y, val_y = train_test_split(
    train_x, train_y, test_size=0.2, stratify=train_y, random_state=42
)

# normalize ด้วย mean/std ของ train set เอง (คำนวณจาก train เท่านั้น กัน data leakage)
DATA_MEAN = train_x.mean()
DATA_STD = train_x.std()
train_x = (train_x - DATA_MEAN) / DATA_STD
val_x = (val_x - DATA_MEAN) / DATA_STD
print(f"mean={DATA_MEAN:.4f}, std={DATA_STD:.4f}")

# converting training images into torch format
train_x = train_x.reshape(-1, 1, IMG_SIZE, IMG_SIZE)  # เราไม่รู้จำนวนที่แน่นอนล่วงหน้าต้องใช้ -1 แทน
train_x = torch.from_numpy(train_x).to(torch.float32)
train_y = torch.from_numpy(train_y).to(torch.long)  # CrossEntropyLoss ต้องการ target เป็น torch.long

# shape of training data
print(train_x.shape, train_y.shape)

# converting validation images into torch format
val_x = val_x.reshape(-1, 1, IMG_SIZE, IMG_SIZE)
val_x = torch.from_numpy(val_x).to(torch.float32)
val_y = torch.from_numpy(val_y).to(torch.long)

# shape of validation data
print(val_x.shape, val_y.shape)

Model Loader

In [ ]:
%%writefile Net.py
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class Net(nn.Module):
    def __init__(self, num_classes=72, dropout=0.3):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # ตัด maxpool ออก: input เล็ก (64x64) ถ้า downsample ตาม conv1+maxpool+4 stage
        # แบบเดิมจะยุบเหลือ 2x2 ก่อนถึง fc รายละเอียดตัวอักษรหายไปเยอะ
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.fc.in_features, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
from Net import Net
model = Net()
print(model)

Defining Learning Algorithm

In [ ]:
# force using 'cuda'
device = torch.device('cuda')

# defining the model
model = Net(num_classes=len(classes)).to(device)

# defining the optimizer (เพิ่ม weight_decay กัน overfit)
optimizer = Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

# LR scheduler: ลด lr ลงครึ่งนึงเมื่อ val_loss ไม่ลดลง 3 epoch ติดกัน
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# class weight กัน class imbalance (คลาสที่มีตัวอย่างเยอะไม่ dominate loss มากไป)
class_counts = torch.bincount(train_y, minlength=len(classes)).float()
class_weights = (class_counts.sum() / (len(classes) * class_counts)).to(device)

# defining the loss function
criterion = CrossEntropyLoss(weight=class_weights)

print(model)

Training Model

In [ ]:
# empty list to store training losses/accuracy
train_losses = []
train_accuracies = []

# empty list to store validation losses/accuracy
val_losses = []
val_accuracies = []

# n_epochs เป็นแค่เพดานบน ให้ early stopping ตัดจบเองตาม val_acc จริง
n_epochs = 60
EARLY_STOP_PATIENCE = 10

from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=64)

best_val_acc = 0.0
epochs_no_improve = 0

for epoch in tqdm(range(n_epochs)):
    model.train()
    tr_loss = 0
    tr_correct = 0
    tr_total = 0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # clearing the Gradients of the model parameters
        optimizer.zero_grad()

        # prediction for training set
        output_train = model(x_batch)

        loss_train = criterion(output_train, y_batch)
        loss_train.backward()
        optimizer.step()
        tr_loss += loss_train.item()

        # accuracy ของ training batch นี้
        predicted = torch.argmax(output_train, dim=1)
        tr_correct += (predicted == y_batch).sum().item()
        tr_total += y_batch.size(0)

    train_losses.append(tr_loss / len(train_loader))
    train_accuracies.append(tr_correct / tr_total)

    # evaluating performance on the validation set
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output_val = model(x_batch)
            val_loss += criterion(output_val, y_batch).item()

            # accuracy ของ validation batch นี้
            predicted = torch.argmax(output_val, dim=1)
            val_correct += (predicted == y_batch).sum().item()
            val_total += y_batch.size(0)

    val_losses.append(val_loss / len(val_loader))
    val_accuracies.append(val_correct / val_total)

    # ลด lr เมื่อ val_loss หยุดลดลง
    scheduler.step(val_losses[-1])

    cur_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_losses[-1]:.4f} - train_acc: {train_accuracies[-1]*100:.2f}% - val_loss: {val_losses[-1]:.4f} - val_acc: {val_accuracies[-1]*100:.2f}% - lr: {cur_lr:.6f}")

    # save เฉพาะตอน val_acc ดีขึ้น (กันเก็บโมเดลที่ overfit ตอนท้าย ๆ)
    if val_accuracies[-1] > best_val_acc:
        best_val_acc = val_accuracies[-1]
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'model.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping ที่ epoch {epoch+1} (val_acc ไม่ดีขึ้น {EARLY_STOP_PATIENCE} epoch ติดกัน) best_val_acc={best_val_acc*100:.2f}%")
            break

Save Model

In [ ]:
# model.pt ถูก save ไว้ระหว่างเทรนแล้วทุกครั้งที่ val_acc ดีขึ้น (checkpoint ที่ val_acc สูงสุด)
# ไม่ต้อง save ซ้ำตรงนี้ — เช็คผลลัพธ์สุดท้ายพอ
print(f"เทรนเสร็จ — best_val_acc: {best_val_acc*100:.2f}% (บันทึกไว้ใน model.pt แล้ว)")